# Plant Disease Detection using CNN
**Dataset:** PlantVillage (Kaggle) — ~87K images, 38 classes  
**Model:** Custom CNN trained with TensorFlow/Keras

## Step 1: Import Libraries

In [50]:
import tensorflow as tf
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import numpy as np
from sklearn.metrics import classification_report

## Step 2: Set Input Parameters

In [51]:
IMAGE_SIZE = (128, 128)
BATCH_SIZE = 32
NUM_CLASSES = 38
EPOCHS = 50
LEARNING_RATE = 0.001

## Step 3: Load and Preprocess Dataset

In [206]:
training_set = tf.keras.utils.image_dataset_from_directory(
    'train',
    image_size=(128, 128),
    batch_size=32,
    color_mode='rgb',
    label_mode='categorical',
    labels='inferred',
    shuffle=True,
    interpolation='bilinear'
)

Found 70295 files belonging to 38 classes.


In [207]:
validation_set = tf.keras.utils.image_dataset_from_directory(
    directory='valid',
    labels='inferred',
    label_mode='categorical',
    batch_size=32,
    image_size=(128, 128),
    color_mode='rgb',
    shuffle=True,
    interpolation='bilinear',
    follow_links=False
)

Found 17572 files belonging to 38 classes.


## Step 4: Define CNN Model Architecture

In [208]:
cnn = tf.keras.models.Sequential()

In [209]:
# Block 1
cnn_model = cnn
cnn_model.add(tf.keras.layers.Conv2D(
    filters=32,
    kernel_size=(3, 3),
    activation='relu',
    padding='same',
    input_shape=(128, 128, 3)
))
cnn_model.add(tf.keras.layers.MaxPooling2D(pool_size=(2, 2), strides=2))

# Block 2
cnn_model.add(tf.keras.layers.Conv2D(
    filters=64,
    kernel_size=(3, 3),
    activation='relu',
    padding='same'
))
cnn_model.add(tf.keras.layers.MaxPooling2D(pool_size=(2, 2)))

# Block 3
cnn_model.add(tf.keras.layers.Conv2D(
    filters=128,
    kernel_size=3,
    activation='relu',
    padding='same'
))
cnn_model.add(tf.keras.layers.MaxPooling2D(pool_size=2, strides=2))

# Block 4
cnn_model.add(tf.keras.layers.Conv2D(
    filters=256,
    kernel_size=3,
    activation='relu',
    padding='same'
))
cnn_model.add(tf.keras.layers.MaxPooling2D(pool_size=2))

# Classifier
cnn_model.add(tf.keras.layers.Flatten())
cnn_model.add(tf.keras.layers.Dense(512, activation='relu'))
cnn_model.add(tf.keras.layers.Dropout(0.4))
cnn_model.add(tf.keras.layers.Dense(38, activation='softmax'))

## Step 5: Compile and Train

In [193]:
cnn.compile(
    optimizer=tf.keras.optimizers.legacy.Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)
cnn.summary()

Model: "sequential_18"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv2d_124 (Conv2D)         (None, 128, 128, 32)      896       
                                                                 
 conv2d_125 (Conv2D)         (None, 126, 126, 32)      9248      
                                                                 
 max_pooling2d_62 (MaxPooli  (None, 63, 63, 32)        0         
 ng2D)                                                           
                                                                 
 conv2d_126 (Conv2D)         (None, 63, 63, 64)        18496     
                                                                 
 conv2d_127 (Conv2D)         (None, 61, 61, 64)        36928     
                                                                 
 max_pooling2d_63 (MaxPooli  (None, 30, 30, 64)        0         
 ng2D)                                               

In [195]:
training_history = cnn.fit(
    x=training_set,
    validation_data=validation_set,
    epochs=50,
    validation_steps=len(validation_set)
)

Epoch 1/50
221/221 [==============================] - 79s 1s/step - loss: 3.350 - accuracy: 0.0965 - val_loss: 0.0635 - val_accuracy: 0.6999
Epoch 2/50
221/221 [==============================] - 244s 110ms/step - loss: 0.001 - accuracy: 0.9981 - val_loss: 0.0626 - val_accuracy: 0.7000
Epoch 3/50
221/221 [==============================] - 27s 120ms/step - loss: 0.3561 - accuracy: 0.9019 - val_loss: 0.2987 - val_accuracy: 0.8073
Epoch 4/50
221/221 [==============================] - 27s 120ms/step - loss: 0.1054 - accuracy: 0.9701 - val_loss: 0.3204 - val_accuracy: 0.8703
Epoch 5/50
221/221 [==============================] - 244s 1s/step - loss: 0.3564 - accuracy: 0.9081 - val_loss: 0.3024 - val_accuracy: 0.8900
Epoch 6/50
221/221 [==============================] - 27s 120ms/step - loss: 0.0636 - accuracy: 0.9028 - val_loss: 0.3002 - val_accuracy: 0.9012
Epoch 7/50
221/221 [==============================] - 27s 120ms/step - loss: 0.1630 - accuracy: 0.9028 - val_loss: 0.3002 - val_accuracy

## Step 6: Evaluate the Model

In [196]:
train_loss, train_acc = cnn.evaluate(training_set, verbose=1)
print('Training accuracy:', train_acc)

1197/1197 [==============================] - 34s 28ms/step - loss: 0.0763 - accuracy: 0.9783
Training accuracy: 0.978310346603424


In [197]:
val_loss, val_acc = cnn.evaluate(validation_set, verbose=1)
print('Validation accuracy:', val_acc)

558/558 [==============================] - 14s 25ms/step - loss: 0.2495 - accuracy: 0.9499
Validation accuracy: 0.949857935831327


## Step 7: Plot Training Curves

In [210]:
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(training_history.history['accuracy'], label='Training Accuracy')
plt.plot(training_history.history['val_accuracy'], label='Validation Accuracy')
plt.title('Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(training_history.history['loss'], label='Training Loss')
plt.plot(training_history.history['val_loss'], label='Validation Loss')
plt.title('Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

plt.tight_layout()
plt.savefig('convergence_curve_plot.png')
plt.show()

## Step 8: Test on a Sample Image

In [37]:
image_path = 'test_sample.jpg'
class_name = training_set.class_names

test_img = tf.keras.preprocessing.image.load_img(
    image_path,
    target_size=(128, 128)
)

img_arr = tf.keras.preprocessing.image.img_to_array(test_img)
img_arr = np.expand_dims(img_arr, axis=0)

prediction = cnn.predict(img_arr)

1/1 [==============================] - 0s 99ms/step


In [48]:
result_index = np.argmax(prediction)
predicted_class = class_name[result_index]

plt.imshow(test_img)
plt.title(f'Predicted Disease: {predicted_class}')
plt.xticks([])
plt.yticks([])
plt.show()

print('Disease Name:', predicted_class)

Disease Name: Apple__Cedar_apple_rust


## Save Model

In [211]:
cnn.save('plant_disease_model.h5')
print('Model saved.')